### 04_synapse_load_scd2_merge_notebook.ipynb
* Synapse Layer — eCourts India Pipeline
* Flow:
    - `bronze/courts`  → Synapse `dim_courts`  (SCD Type 2)
    - `silver/silver_cleaned` → Synapse `fact_cases`  (MERGE on case_id)

### `IMPORTS `

In [0]:
import pyodbc
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, lit
from pyspark.sql.types import DateType
from pyspark.sql.functions import *

### `SPARK Session`

In [0]:
spark = SparkSession.builder \
                .config('spark.sql.shuffle.partitions', '20')     \
                .getOrCreate()

In [0]:
bronze_courts_path  = "abfss://bronze@saadlsecourtsindia.dfs.core.windows.net/courts/"
silver_cleaned_path = "abfss://silver@saadlsecourtsindia.dfs.core.windows.net/cases_cleaned/"

### `ADLS AUTH`

In [0]:
# ==================================ADLS AUTH=======================================
storage_account = "saadlsecourtsindia"
storage_key    = dbutils.secrets.get(scope="kv-ecourtsproject", key="adls-storage-key")
client_id      = dbutils.secrets.get(scope="kv-ecourtsproject", key="sp-client-id")
client_secret  = dbutils.secrets.get(scope="kv-ecourtsproject", key="sp-client-secret")
tenant_id      = dbutils.secrets.get(scope="kv-ecourtsproject", key="sp-tenant-id")


spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",storage_key)
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")


### `SYNAPSE CONN`

In [0]:
synapse_password = dbutils.secrets.get(scope="kv-ecourtsproject", key="synapse-db-password-sqlamin")

synapse_url = (
    "jdbc:sqlserver://ws-synapse-ecourtsindia.sql.azuresynapse.net:1433;"
    "database=sqlpool_ecourtsindia;"
    "user=sqladmin;"
    f"password={synapse_password};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.sql.azuresynapse.net;"
    "loginTimeout=30;"
)

synapse_temp_dir = "abfss://synapse@saadlsecourtsindia.dfs.core.windows.net/temp/"

# pyodbc connection string — single definition, reused everywhere
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=ws-synapse-ecourtsindia.sql.azuresynapse.net;"
    "DATABASE=sqlpool_ecourtsindia;"
    "UID=sqladmin;"
    f"PWD={synapse_password};"
)

### ` `

In [0]:
def synapse_read(table_name):
    return (
        spark.read
        .format("com.databricks.spark.sqldw")
        .option("url", synapse_url)
        .option("tempDir", synapse_temp_dir)
        .option("forwardSparkAzureStorageCredentials", "true")
        .option("dbTable", table_name)
        .load()
    )


In [0]:
def synapse_write(df, table_name, mode="append"):
    (
        df.write
        .format("com.databricks.spark.sqldw")
        .option("url", synapse_url)
        .option("tempDir", synapse_temp_dir)
        .option("forwardSparkAzureStorageCredentials", "true")
        .option("dbTable", table_name)
        .mode(mode)
        .save()
    )

In [0]:
def run_sql(sql):
    conn = spark.sparkContext._gateway.jvm.java.sql.DriverManager.getConnection(synapse_url)
    stmt = conn.createStatement()
    stmt.execute(sql)
    conn.close()

In [0]:
# ==================================READ SOURCES====================================
courts_df = spark.read.format("delta").load(bronze_courts_path)
cases_df  = spark.read.format("delta").load(silver_cleaned_path)

print(f"bronze/courts count  : {courts_df.count()}")
print(f"silver_cleaned count : {cases_df.count()}")

# Casueed the schema Mismacthed error, also has no value in the fact table
cases_df = cases_df.drop("bronze_ingested_at", "source_file")

bronze/courts count  : 50
silver_cleaned count : 1000


In [0]:
# ==================================DIM_COURTS SCD2=================================
print("\n--- dim_courts SCD2 ---")

# check if dim_courts exists in Synapse
try:
    existing_dim = synapse_read("dim_courts")
    dim_exists   = existing_dim.limit(1).count() > 0
    print(f"dim_courts exists, proceeding with SCD2")
except Exception as e:
    dim_exists = False
    print(f"dim_courts not found, doing initial load : {e}")

if not dim_exists:
    # First run — load all courts as current records
    dim_initial = (
        courts_df
        .withColumn("valid_from", current_date())
        .withColumn("valid_to",   lit("9999-12-31").cast(DateType()))
        .withColumn("is_current", lit(True))
    )
    synapse_write(dim_initial, "dim_courts", mode="append")
    print("dim_courts : initial load complete")

else:
    # Step 1 — find changed records (judge_count changed)
    current_dim = existing_dim.filter(col("is_current") == True)
    
    changed_ids = courts_df.alias("src").join(
        current_dim.alias("tgt"), on="court_id", how="inner"
    ).filter(
        col("src.judge_count") != col("tgt.judge_count")
    ).select("src.court_id")

    changed_list = [r.court_id for r in changed_ids.collect()]

    # Step 2 — expire changed records in existing dim
    updated_dim = existing_dim \
        .withColumn("is_current", \
            when(col("court_id").isin(changed_list), lit(False)) \
            .otherwise(col("is_current"))
        ) \
        .withColumn("valid_to",
            when(col("court_id").isin(changed_list), current_date())
            .otherwise(col("valid_to"))
        )

    # Step 3 — new versions for changed + brand new courts
    new_versions = courts_df.join(
        current_dim.select("court_id", "judge_count"),
        on="court_id", how="left_anti"
    ).withColumn("valid_from", current_date()) \
     .withColumn("valid_to",   lit("9999-12-31").cast(DateType())) \
     .withColumn("is_current", lit(True))

    # Step 4 — combine and overwrite dim_courts
    final_dim = updated_dim.unionByName(new_versions, allowMissingColumns=True)
    synapse_write(final_dim, "dim_courts", mode="overwrite")
    print(f"dim_courts : SCD2 complete — {len(changed_list)} records expired")


    print("dim_courts : SCD2 expire + insert complete")




--- dim_courts SCD2 ---
dim_courts exists, proceeding with SCD2
dim_courts : SCD2 complete — 0 records expired
dim_courts : SCD2 expire + insert complete


In [0]:
# ==================================FACT_CASES MERGE================================
print("\n--- fact_cases MERGE ---")

# check if fact_cases exists
try:
    existing_fact = synapse_read("fact_cases")
    fact_exists   = existing_fact.limit(1).count() > 0
    print(f"fact_cases exists, proceeding with MERGE")
except Exception as e:
    fact_exists = False
    print(f"fact_cases not found, doing initial load : {e}")

if not fact_exists:
    # First run — insert all cases
    synapse_write(cases_df, "fact_cases", mode="append")
    print("fact_cases : initial load complete")

else:
    existing_ids = [row.case_id for row in existing_fact.select("case_id").collect()]
    
    # unchanged cases already in Synapse but not in incoming batch
    unchanged = existing_fact.filter(~col("case_id").isin(existing_ids))
    
    # combine unchanged existing + all incoming (handles both updates and new inserts)
    final_fact = unchanged.unionByName(cases_df, allowMissingColumns=True)
    
    synapse_write(final_fact, "fact_cases", mode="overwrite")
    print("fact_cases : MERGE complete")


--- fact_cases MERGE ---
fact_cases exists, proceeding with MERGE
fact_cases : MERGE complete


### `VALIDATION`

In [0]:
# ==================================VALIDATION======================================
print("\n=== VALIDATION ===")

print("dim_courts current records :")
synapse_read("dim_courts") \
    .filter(col("is_current") == True) \
    .show(20, truncate=False)

dim_total    = synapse_read("dim_courts").count()
dim_current  = synapse_read("dim_courts").filter(col("is_current") == True).count()
dim_expired  = dim_total - dim_current
print(f"dim_courts  — total: {dim_total} | current: {dim_current} | expired: {dim_expired}")

fact_total = synapse_read("fact_cases").count()
fact_active = synapse_read("fact_cases").filter(col("is_deleted") == False).count()
print(f"fact_cases  — total: {fact_total} | active: {fact_active} | deleted: {fact_total - fact_active}")